# DQN vs PPO — Comparative Analysis

This notebook runs both algorithms on the same environment with identical seeds, 
collects training curves, and produces side-by-side diagnostic plots.

## Algorithm Summary

| Property | DQN | PPO |
|---|---|---|
| Type | Off-policy, value-based | On-policy, policy gradient |
| Data reuse | Yes (replay buffer) | No (rollout discarded) |
| Policy | Implicit (ε-greedy over Q) | Explicit (softmax policy) |
| Variance reduction | Target network + Double DQN | GAE + advantage normalization |
| Constraint mechanism | None (optimism bias) | Clipped ratio |
| Typical sample efficiency | Higher (off-policy) | Lower (on-policy) |
| Typical stability | Moderate | High |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import torch
import gymnasium as gym

from src.agents.dqn import DQNAgent
from src.agents.ppo import PPOAgent
from src.environments.utils import get_env_dims, set_global_seed
from src.environments.wrappers import make_env
from src.evaluation.evaluator import Evaluator
from src.evaluation.plotting import plot_comparison, plot_training_curve, smooth
from src.training.trainer import DQNTrainer, PPOTrainer

SEED = 42
ENV_ID = 'CartPole-v1'  # Change to 'LunarLander-v3' for harder task
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## Train DQN

In [ ]:
set_global_seed(SEED)
dqn_env = make_env(ENV_ID, seed=SEED)
obs_dim, action_dim = get_env_dims(dqn_env)

dqn_agent = DQNAgent(
    obs_dim=obs_dim,
    action_dim=action_dim,
    hidden_dims=(256, 256),
    lr=1e-4,
    gamma=0.99,
    epsilon_decay_steps=50_000,
    buffer_capacity=50_000,
    batch_size=64,
    target_update_freq=500,
    dueling=True,
    double_dqn=True,
    device=DEVICE,
)

dqn_trainer = DQNTrainer(
    agent=dqn_agent,
    env=dqn_env,
    total_timesteps=200_000,
    log_interval=10_000,
    checkpoint_dir=Path('../checkpoints/notebook/dqn'),
    verbose=True,
)
dqn_metrics = dqn_trainer.train()
dqn_env.close()
print(f'DQN training complete. Episodes: {len(dqn_metrics.episode_rewards)}')

## Train PPO

In [ ]:
set_global_seed(SEED)
ppo_env = make_env(ENV_ID, seed=SEED)

ppo_agent = PPOAgent(
    obs_dim=obs_dim,
    action_dim=action_dim,
    hidden_dims=(64, 64),
    lr=2.5e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_coef=0.2,
    vf_coef=0.5,
    ent_coef=0.01,
    num_steps=512,
    num_minibatches=8,
    update_epochs=10,
    total_timesteps=500_000,
    device=DEVICE,
)

ppo_trainer = PPOTrainer(
    agent=ppo_agent,
    env=ppo_env,
    total_timesteps=500_000,
    checkpoint_dir=Path('../checkpoints/notebook/ppo'),
    verbose=True,
)
ppo_metrics = ppo_trainer.train()
ppo_env.close()
print(f'PPO training complete. Episodes: {len(ppo_metrics.episode_rewards)}')

## Training Curve Comparison

In [ ]:
fig = plot_comparison(
    {'DQN (Double + Dueling)': dqn_metrics.episode_rewards,
     'PPO (GAE + Clip)': ppo_metrics.episode_rewards},
    title=f'DQN vs PPO — {ENV_ID}',
    smoothing_window=20,
)
plt.show()

## Evaluation — Greedy Policy (No Exploration)

In [ ]:
eval_env = gym.make(ENV_ID)
evaluator = Evaluator(eval_env, n_episodes=50)

dqn_stats = evaluator.evaluate(dqn_agent)
ppo_stats = evaluator.evaluate(ppo_agent)
eval_env.close()

print('=' * 50)
print(f'{'Metric':<20} {'DQN':>12} {'PPO':>12}')
print('-' * 50)
for key in dqn_stats:
    print(f'{key:<20} {dqn_stats[key]:>12.2f} {ppo_stats[key]:>12.2f}')
print('=' * 50)

## Sample Efficiency: Return vs Environment Steps

This is the canonical comparison metric in RL research. Both axes are normalized:
- X: cumulative environment steps (not episodes, not wall time)
- Y: smoothed return


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

# DQN: steps are recorded per episode in metrics.steps
if dqn_metrics.steps:
    dqn_smoothed = smooth(dqn_metrics.episode_rewards, 20)
    offset = len(dqn_metrics.episode_rewards) - len(dqn_smoothed)
    ax.plot(
        dqn_metrics.steps[offset:],
        dqn_smoothed,
        color='#58a6ff',
        label='DQN',
        linewidth=2,
    )

if ppo_metrics.steps:
    ppo_smoothed = smooth(ppo_metrics.episode_rewards, 20)
    offset = len(ppo_metrics.episode_rewards) - len(ppo_smoothed)
    ax.plot(
        ppo_metrics.steps[offset:],
        ppo_smoothed,
        color='#3fb950',
        label='PPO',
        linewidth=2,
    )

ax.axhline(475, color='#f78166', linestyle='--', alpha=0.7, label='Solved threshold (475)')
ax.set_xlabel('Environment steps', color='#c9d1d9')
ax.set_ylabel('Return (EMA-20)', color='#c9d1d9')
ax.set_title(f'Sample Efficiency — {ENV_ID}', color='#c9d1d9')
ax.tick_params(colors='#8b949e')
ax.legend()
ax.grid(True, color='#21262d', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## Key Observations

**DQN characteristics:**
- Off-policy: reuses past experience via replay buffer — more sample efficient early on
- Double DQN mitigates Q-value overestimation, preventing premature convergence to suboptimal policies
- Dueling architecture learns state values independently from action advantages
- Susceptible to instability from the non-stationarity of the target

**PPO characteristics:**
- On-policy: each rollout is used for one update then discarded
- The clipped objective provides a trust region without expensive second-order computation
- GAE (λ=0.95) balances bias and variance in advantage estimates
- Tends to be more monotonically improving but requires more total steps

**When to choose which:**
- Prefer DQN when sample efficiency matters (expensive simulator, slow environment)
- Prefer PPO when stability matters or you have a fast environment
- For continuous action spaces, PPO generalizes directly; DQN requires discretization or actor-critic variants (DDPG, TD3, SAC)